In [2]:
# -- Imports --
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path
from scipy.stats import pearsonr
from sklearn.decomposition import PCA

In [3]:
# -- Variables --
PROJECT_ROOT  = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "data" / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

In [6]:
# Load feature matrix and validation data
X  = pd.read_parquet(PROCESSED_DIR / "feature_matrix_scaled.parquet")
le = pd.read_parquet(PROCESSED_DIR / "life_expectancy_validation.parquet")

print(f"✅ Feature matrix: {X.shape}")
print(f"✅ Life expectancy: {le.shape}")
print(f"\nFeatures:\n{X.columns.tolist()}")

✅ Feature matrix: (509, 27)
✅ Life expectancy: (509, 2)

Features:
['unemployment_rate_mean', 'unemployment_rate_slope', 'poverty_risk_mean', 'poverty_risk_slope', 'internet_usage_mean', 'internet_usage_slope', 'gdp_per_capita_mean', 'gdp_per_capita_slope', 'social_trust_mean', 'social_trust_slope', 'life_satisfaction_mean', 'life_satisfaction_slope', 'aerobic_activity_mean', 'aerobic_activity_slope', 'heavy_drinking_mean', 'heavy_drinking_slope', 'smoking_mean', 'smoking_slope', 'fruit_veggies_mean', 'fruit_veggies_slope', 'social_contact_mean', 'social_contact_slope', 'family_contact_mean', 'family_contact_slope', 'obesity_rate_mean', 'obesity_rate_slope', 'social_support_mean']


In [7]:
pca = PCA()
pca.fit(X)

explained   = pd.Series(
    pca.explained_variance_ratio_ * 100,
    index=[f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))]
)
cumulative = explained.cumsum()

print("Explained variance per component:")
print(explained.round(2).to_string())
print(f"\nPC1 explains:        {explained.iloc[0]:.1f}%")
print(f"PC1+PC2 explain:     {cumulative.iloc[1]:.1f}%")
print(f"PC1+PC2+PC3 explain: {cumulative.iloc[2]:.1f}%")

Explained variance per component:
PC1     28.31
PC2     12.15
PC3     10.48
PC4      7.02
PC5      6.48
PC6      5.66
PC7      4.35
PC8      4.22
PC9      3.83
PC10     2.85
PC11     2.38
PC12     2.18
PC13     1.91
PC14     1.61
PC15     1.27
PC16     1.09
PC17     0.87
PC18     0.69
PC19     0.68
PC20     0.43
PC21     0.40
PC22     0.34
PC23     0.27
PC24     0.19
PC25     0.13
PC26     0.12
PC27     0.09

PC1 explains:        28.3%
PC1+PC2 explain:     40.5%
PC1+PC2+PC3 explain: 50.9%


In [8]:
loadings = pd.Series(
    pca.components_[0],
    index=X.columns
).sort_values(key=abs, ascending=False)

print("PC1 loadings (sorted by absolute value):")
print(loadings.round(4).to_string())

PC1 loadings (sorted by absolute value):
internet_usage_mean        0.3241
aerobic_activity_mean      0.3156
internet_usage_slope      -0.3040
smoking_mean              -0.2969
life_satisfaction_mean     0.2881
heavy_drinking_mean       -0.2718
obesity_rate_slope         0.2499
gdp_per_capita_mean        0.2449
unemployment_rate_mean    -0.2134
poverty_risk_mean         -0.2110
social_trust_mean          0.2076
life_satisfaction_slope   -0.1939
fruit_veggies_mean         0.1855
family_contact_slope      -0.1581
family_contact_mean        0.1413
unemployment_rate_slope    0.1406
social_contact_mean        0.1205
social_contact_slope      -0.1179
heavy_drinking_slope      -0.1061
social_trust_slope        -0.0842
gdp_per_capita_slope      -0.0828
smoking_slope              0.0619
poverty_risk_slope         0.0525
obesity_rate_mean          0.0450
social_support_mean        0.0397
fruit_veggies_slope        0.0393
aerobic_activity_slope     0.0165


In [9]:
scores = pca.transform(X)[:, 0]
r, p = pearsonr(scores, le["life_expectancy_mean"])
print(f"\nr(PC1, life expectancy) = {r:.3f}  p={p:.4f}")
if r < 0:
    print("→ Sign flip needed — PC1 points away from flourishing")
else:
    print("→ Sign correct — PC1 points toward flourishing")


r(PC1, life expectancy) = 0.505  p=0.0000
→ Sign correct — PC1 points toward flourishing


In [10]:
for i in range(4):
    pc_scores = pca.transform(X)[:, i]
    r, p = pearsonr(pc_scores, le["life_expectancy_mean"])
    print(f"PC{i+1}: r = {r:.3f}  p = {p:.4f}")

PC1: r = 0.505  p = 0.0000
PC2: r = -0.263  p = 0.0000
PC3: r = 0.443  p = 0.0000
PC4: r = -0.009  p = 0.8435


In [11]:
# Mean features only
mean_cols = [c for c in X.columns if c.endswith("_mean")]
X_mean_only = X[mean_cols]

pca_mean = PCA()
pca_mean.fit(X_mean_only)
scores_mean = pca_mean.transform(X_mean_only)[:, 0]
r_mean, _ = pearsonr(scores_mean, le["life_expectancy_mean"])
print(f"\nPCA on mean features only:")
print(f"PC1 explains: {pca_mean.explained_variance_ratio_[0]*100:.1f}%")
print(f"r(PC1, life expectancy) = {r_mean:.3f}")


PCA on mean features only:
PC1 explains: 41.5%
r(PC1, life expectancy) = 0.518


In [12]:
from sklearn.cross_decomposition import PLSRegression
pls = PLSRegression(n_components=1)
pls.fit(X, le["life_expectancy_mean"])
scores_pls = pls.transform(X)
r_pls, _ = pearsonr(scores_pls[:,0], le["life_expectancy_mean"])
print(f"PLS r = {r_pls:.3f}")

PLS r = 0.711


In [13]:
r_ls, _ = pearsonr(scores_pls[:,0], le["life_expectancy_mean"])
# Already know this: 0.711

# Now validate against life satisfaction (true holdout)
ls = pd.read_parquet(PROCESSED_DIR / "life_expectancy_validation.parquet")

# Load life satisfaction separately
ls_interim = pd.read_parquet(PROJECT_ROOT / "data" / "interim" / "life_satisfaction.parquet")